# Module 3, Class 3 Assignment: Categorical Encoding

**Student:** Firdavs Boliev

This notebook completes the Module 3 Class 3 homework in a beginner-friendly way. The code cells include clear comments that explain what each step is doing.

## What this notebook covers
- Found the text-based categorical columns in the Telco dataset.
- Encoded Contract with pandas get_dummies.
- Used drop_first to avoid keeping redundant dummy columns.
- Used sklearn OneHotEncoder and mapped Churn into a numeric target.


In [1]:
# This setup cell imports the libraries, loads the Telco dataset, and prepares TotalCharges as a numeric feature.
# === SETUP — run this first ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
print('Loaded:', df.shape)


Loaded: (7043, 21)


### Cell 1 — see which columns are categorical (text)

In [2]:
# This cell finds all columns that contain text categories instead of numeric values.
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Categorical columns:', cat_cols)
print('Total:', len(cat_cols))


Categorical columns: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']
Total: 17


### Cell 2 — look at one categorical column

In [3]:
# This cell counts the values in Contract so we know which categories will be encoded.
df['Contract'].value_counts()


Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

### Cell 3 — use pd.get_dummies to one-hot encode Contract

In [4]:
# This cell uses pandas get_dummies to turn the Contract text values into separate 0/1 columns.
dummies = pd.get_dummies(df['Contract'], prefix='Contract')
print('Shape before:', df[['Contract']].shape, '-> after:', dummies.shape)
dummies.head()


Shape before: (7043, 1) -> after: (7043, 3)


,Contract_Month-to-month,Contract_One year,Contract_Two year
0,True,False,False
1,False,True,False
2,True,False,False
3,False,True,False
4,True,False,False


### Cell 4 — drop_first=True (to avoid the dummy variable trap)

In [5]:
# This cell repeats one-hot encoding with drop_first=True to remove one redundant category column.
dummies2 = pd.get_dummies(df['Contract'], prefix='Contract', drop_first=True)
print('With drop_first:', dummies2.shape)
dummies2.head()


With drop_first: (7043, 2)


,Contract_One year,Contract_Two year
0,False,False
1,True,False
2,False,False
3,True,False
4,False,False


### Cell 5 — use OneHotEncoder from sklearn

In [6]:
# This cell uses sklearn OneHotEncoder to encode PaymentMethod and then stores the result in a DataFrame.
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)
encoded = ohe.fit_transform(df[['PaymentMethod']])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(['PaymentMethod']))
encoded_df.head()


,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0.0,0.0,1.0,0.0
1,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0
4,0.0,0.0,1.0,0.0


### Cell 6 — binary Yes/No → 1/0 mapping

In [7]:
# This cell maps the Churn target from Yes and No text into 1 and 0 numbers.
df['Churn_bin'] = df['Churn'].map({'No': 0, 'Yes': 1})
df[['Churn', 'Churn_bin']].head()


,Churn,Churn_bin
0,No,0
1,No,0
2,Yes,1
3,No,0
4,Yes,1


### Cell 7 — encode multiple columns at once

In [8]:
# This cell one-hot encodes several categorical columns at the same time for model-ready features.
multi = pd.get_dummies(df[['gender', 'Contract', 'PaymentMethod']], drop_first=True)
print('Shape:', multi.shape)
multi.head()


Shape: (7043, 6)


,gender_Male,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,False,False,False,False,True,False
1,True,True,False,False,False,True
2,True,False,False,False,False,True
3,True,True,False,False,False,False
4,False,False,False,False,True,False
